# Content Pipeline

Runs the six JSON/dict applications from collection through YouTube metadata.
Voiceover, ChatGPT handoffs, and visual production have separate notebooks.


In [ ]:
from __future__ import annotations

from copy import deepcopy
from datetime import datetime, timedelta, timezone
import json
from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "pipeline" / "apps").exists():
    raise RuntimeError("Run this notebook from the repository root or output directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline.orchestration import create_run_context, module_output_dir

APPS_DIR = PROJECT_ROOT / "src" / "pipeline" / "apps"
NEWS_COLLECTION_DIR = APPS_DIR / "news_collection"
PIPELINE_CONFIG_PATH = PROJECT_ROOT / "src" / "pipeline" / "config" / "workbench.yaml"
OUTPUT_ROOT = PROJECT_ROOT / "output"
IST = timezone(timedelta(hours=5, minutes=30))

print(f"Project root: {PROJECT_ROOT}")
print(f"Apps dir:     {APPS_DIR}")
print(f"Output root:  {OUTPUT_ROOT}")
print(f"Config:       {PIPELINE_CONFIG_PATH}")


In [ ]:
# Set RUN_PROFILE to None to use workbench.yaml's default.
# Set RESUME_RUN_ID only when intentionally continuing an existing run.
RUN_PROFILE = None
RESUME_RUN_ID = None
METADATA_SCRIPT_RESULT_PATH = None  # Optional canonical ChatGPT script result.

RUN_CONTEXT = create_run_context(
    project_root=PROJECT_ROOT,
    run_profile=RUN_PROFILE,
    run_id=RESUME_RUN_ID,
    trigger="notebook",
    orchestrator="notebooks/production/content_pipeline.ipynb",
    config_path=PIPELINE_CONFIG_PATH,
    output_root=OUTPUT_ROOT,
)
RUN_ID = RUN_CONTEXT["run_id"]
RUN_PROFILE = RUN_CONTEXT["run_profile"]
RUN_DIR = Path(RUN_CONTEXT["run_dir"])
PROFILE_CONFIG = deepcopy(RUN_CONTEXT["profile_config"])

NEWS_OUTPUT_DIR = module_output_dir(RUN_CONTEXT, "news_collection")
EVIDENCE_OUTPUT_DIR = module_output_dir(RUN_CONTEXT, "evidence_prep")
STORY_OUTPUT_DIR = module_output_dir(RUN_CONTEXT, "story_mapping")
VIDEO_OUTPUT_DIR = module_output_dir(RUN_CONTEXT, "video_packaging")
SCRIPT_OUTPUT_DIR = module_output_dir(RUN_CONTEXT, "script_generation")
METADATA_OUTPUT_DIR = module_output_dir(RUN_CONTEXT, "metadata_creator")

print(f"Run profile: {RUN_PROFILE}")
print(f"Run ID:      {RUN_ID}")
print(f"Run dir:     {RUN_DIR}")
print(
    "LLMs:        "
    f"story_mapping={PROFILE_CONFIG['story_mapping']['story_label']['provider']}/{PROFILE_CONFIG['story_mapping']['story_label']['model']}, "
    f"script_generation={PROFILE_CONFIG['script_generation']['provider']}/{PROFILE_CONFIG['script_generation']['model']}, "
    f"metadata_creator={PROFILE_CONFIG['metadata_creator']['provider']}/{PROFILE_CONFIG['metadata_creator']['model']}"
)


In [ ]:
def load_yaml(path: Path) -> dict:
    data = yaml.safe_load(path.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise ValueError(f"Expected YAML object at {path}")
    return data


def workbench_module_config(module_name: str) -> dict:
    config = deepcopy(PROFILE_CONFIG.get(module_name, {}))
    if not config:
        raise KeyError(f"Profile {RUN_PROFILE!r} is missing config for {module_name!r}")
    return config


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    text = "\n".join(json.dumps(row, ensure_ascii=False) for row in rows)
    if text:
        text += "\n"
    path.write_text(text, encoding="utf-8")


def require_app_ready(result: dict, app_name: str, *, allowed_statuses=("ok",), handoff_key=None) -> None:
    status = result.get("status")
    if status not in allowed_statuses:
        summary = result.get("summary", {})
        failure_reason = summary.get("failure_reason") or summary.get("error") or result.get("reason") or "No failure reason provided."
        raise RuntimeError(f"{app_name} is not ready. status={status!r}. reason={failure_reason}")
    if handoff_key and not bool(result.get("handoff", {}).get(handoff_key, False)):
        summary = result.get("summary", {})
        failure_reason = summary.get("failure_reason") or summary.get("error") or f"handoff.{handoff_key} is false"
        raise RuntimeError(f"{app_name} is not ready. status={status!r}. reason={failure_reason}")


def require_non_empty_script(script_generation_result: dict) -> str:
    require_app_ready(script_generation_result, "script_generation", handoff_key="ready_for_metadata_creation")
    script_package = script_generation_result.get("data", {}).get("script_package", {})
    full_script = str(script_package.get("full_script", "")).strip()
    if not full_script:
        checks = script_package.get("checks", {})
        errors = checks.get("errors") or []
        reason = "; ".join(str(error) for error in errors) or "script_package.full_script is empty"
        raise RuntimeError(f"script_generation did not produce a non-empty full_script. reason={reason}")
    return full_script


def collection_window_for_profile(profile: dict) -> dict:
    end = datetime.now(IST)
    lookback_hours = int(profile["lookback_hours"])
    start = end - timedelta(hours=lookback_hours)
    return {
        "start": start.isoformat(),
        "end": end.isoformat(),
        "lookback_hours": lookback_hours,
    }


def build_news_collection_input(run_id: str, run_profile: str) -> dict:
    profile_config = load_yaml(NEWS_COLLECTION_DIR / "profiles.yaml")
    source_config = load_yaml(NEWS_COLLECTION_DIR / "source_config.yaml")
    profiles = profile_config["profiles"]
    if run_profile not in profiles:
        raise KeyError(f"Unknown run profile: {run_profile}")

    profile = profiles[run_profile]
    source_groups = source_config["source_groups"]
    window = collection_window_for_profile(profile)
    collectors = {}
    for group_name in profile["enabled_source_groups"]:
        collector_config = deepcopy(source_groups[group_name])
        collector_config["source_group"] = group_name
        collector_config["collection_window"] = window
        collector_config["query_themes"] = list(profile["query_themes"])
        collector_config["runtime"] = deepcopy(profile.get("collector_caps", {}).get(group_name, {}))
        collectors[group_name] = collector_config

    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": {
            "enabled_source_groups": list(profile["enabled_source_groups"]),
            "query_themes": list(profile["query_themes"]),
            "collectors": collectors,
        },
        "input": {
            "run_profile": run_profile,
            "channel": deepcopy(profile_config["channel"]),
        },
    }


def build_evidence_prep_input(run_id: str, news_input: dict, news_result: dict) -> dict:
    first_collector = next(iter(news_input["config"]["collectors"].values()))
    config = workbench_module_config("evidence_prep")
    config["collection_window"] = deepcopy(first_collector["collection_window"])
    config["query_themes"] = list(news_input["config"]["query_themes"])
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": config,
        "input": {
            "news_collection": news_result,
        },
    }


def build_story_mapping_input(run_id: str, evidence_input: dict, evidence_result: dict) -> dict:
    config = workbench_module_config("story_mapping")
    config["collection_window"] = deepcopy(evidence_input["config"]["collection_window"])
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": config,
        "input": {
            "evidence_prep": evidence_result,
        },
    }


def build_video_packaging_input(run_id: str, story_mapping_result: dict) -> dict:
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": workbench_module_config("video_packaging"),
        "input": {
            "story_mapping": story_mapping_result,
        },
    }


def build_script_generation_input(run_id: str, video_packaging_result: dict) -> dict:
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": workbench_module_config("script_generation"),
        "input": {
            "video_packaging": video_packaging_result,
        },
    }


def build_metadata_creator_input(run_id: str, script_generation_result: dict, video_packaging_result: dict) -> dict:
    profile_config = load_yaml(NEWS_COLLECTION_DIR / "profiles.yaml")
    channel = deepcopy(profile_config.get("channel", {}))
    config = workbench_module_config("metadata_creator")
    config["channel"] = {
        "channel_name": channel.get("channel_name", "ManiVerse - AI"),
        "content_promise": channel.get("content_promise", ""),
    }
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": config,
        "input": {
            "script_generation": script_generation_result,
            "video_packaging": video_packaging_result,
        },
    }


## App 1: News Collection


In [ ]:
from src.pipeline.apps.news_collection import run as run_news_collection

news_input = build_news_collection_input(RUN_ID, RUN_PROFILE)
news_result = run_news_collection(news_input)
require_app_ready(news_result, "news_collection", allowed_statuses=("ok", "partial"))
write_json(NEWS_OUTPUT_DIR / "news_collection_input.json", news_input)

write_json(NEWS_OUTPUT_DIR / "news_collection_result.json", news_result)
write_json(NEWS_OUTPUT_DIR / "raw_records.json", news_result["data"])
write_json(NEWS_OUTPUT_DIR / "summary.json", news_result["summary"])

print(f"News collection status: {news_result['status']}")
print(f"Raw records:            {news_result['data']['record_count']}")
print(f"Wrote:                  {NEWS_OUTPUT_DIR}")


In [ ]:
news_result["summary"]


## App 2: Evidence Prep


In [ ]:
from src.pipeline.apps.evidence_prep import run as run_evidence_prep

evidence_input = build_evidence_prep_input(RUN_ID, news_input, news_result)
evidence_result = run_evidence_prep(evidence_input)
require_app_ready(evidence_result, "evidence_prep")
write_json(EVIDENCE_OUTPUT_DIR / "evidence_prep_input.json", evidence_input)

write_json(EVIDENCE_OUTPUT_DIR / "evidence_prep_result.json", evidence_result)
write_json(EVIDENCE_OUTPUT_DIR / "prepared_records.json", evidence_result["data"])
write_json(EVIDENCE_OUTPUT_DIR / "summary.json", evidence_result["summary"])

print(f"Evidence prep status: {evidence_result['status']}")
print(f"Prepared records:     {evidence_result['data']['record_count']}")
print(f"Wrote:                {EVIDENCE_OUTPUT_DIR}")


In [ ]:
evidence_result["summary"]


## App 3: Story Mapping


In [ ]:
# Optional story-mapping resume helper:
# Keep this commented for normal full-notebook runs.
#
# import json
# RESUME_RUN_ID = "run_YYYY_MM_DD_HHMMSS_daily_48h_all_sources"
# RUN_ID = RESUME_RUN_ID
# with open(RUN_DIR / "evidence_prep" / "evidence_prep_result.json", encoding="utf-8") as f:
#     evidence_result = json.load(f)


In [ ]:
from src.pipeline.apps.story_mapping import run as run_story_mapping

story_mapping_input = build_story_mapping_input(RUN_ID, evidence_input, evidence_result)
story_mapping_result = run_story_mapping(story_mapping_input)
require_app_ready(story_mapping_result, "story_mapping", handoff_key="ready_for_video_packaging")
write_json(STORY_OUTPUT_DIR / "story_mapping_input.json", story_mapping_input)

write_json(STORY_OUTPUT_DIR / "story_mapping_result.json", story_mapping_result)
write_json(STORY_OUTPUT_DIR / "stories.json", story_mapping_result["data"])
write_json(STORY_OUTPUT_DIR / "summary.json", story_mapping_result["summary"])
write_jsonl(STORY_OUTPUT_DIR / "llm_calls.jsonl", story_mapping_result["llm_calls"])

print(f"Story mapping status: {story_mapping_result['status']}")
print(f"Stories:              {story_mapping_result['data']['story_count']}")
print(f"Wrote:                {STORY_OUTPUT_DIR}")


In [ ]:
story_mapping_result["summary"]


## App 4: Video Packaging


In [ ]:
# Optional video-packaging resume helper:
# Keep this commented for normal full-notebook runs.
#
# import json
# RESUME_RUN_ID = "run_YYYY_MM_DD_HHMMSS_daily_48h_all_sources"
# RUN_ID = RESUME_RUN_ID
# with open(RUN_DIR / "story_mapping" / "story_mapping_result.json", encoding="utf-8") as f:
#     story_mapping_result = json.load(f)


In [ ]:
from src.pipeline.apps.video_packaging import run as run_video_packaging

video_packaging_input = build_video_packaging_input(RUN_ID, story_mapping_result)
video_packaging_result = run_video_packaging(video_packaging_input)
require_app_ready(video_packaging_result, "video_packaging", handoff_key="ready_for_script_generation")
write_json(VIDEO_OUTPUT_DIR / "video_packaging_input.json", video_packaging_input)

write_json(VIDEO_OUTPUT_DIR / "video_packaging_result.json", video_packaging_result)
write_json(VIDEO_OUTPUT_DIR / "video_packet.json", video_packaging_result["data"]["video_packet"])
write_json(VIDEO_OUTPUT_DIR / "summary.json", video_packaging_result["summary"])

print(f"Video packaging status: {video_packaging_result['status']}")
print(f"Selected format:        {video_packaging_result['data']['selected_format']}")
print(f"Selected stories:       {video_packaging_result['summary']['selected_story_count']}")
print(f"Ready for script gen:   {video_packaging_result['handoff']['ready_for_script_generation']}")
print(f"Wrote:                  {VIDEO_OUTPUT_DIR}")


In [ ]:
video_packaging_result["summary"]


## App 5: Script Generation


In [ ]:
# Optional resume helper:
# If you want to start from an existing video_packaging result, set RESUME_RUN_ID
# manually and load it here. Keep this commented for normal full-notebook runs.
#
# import json
# RESUME_RUN_ID = "run_YYYY_MM_DD_HHMMSS_daily_48h_all_sources"
# RUN_ID = RESUME_RUN_ID
# with open(RUN_DIR / "video_packaging" / "video_packaging_result.json", encoding="utf-8") as f:
#     video_packaging_result = json.load(f)


In [ ]:
from src.pipeline.apps.script_generation.formats.roundup import run as run_roundup_script_generation

script_generation_input = build_script_generation_input(RUN_ID, video_packaging_result)
script_generation_result = run_roundup_script_generation(script_generation_input)
require_non_empty_script(script_generation_result)
write_json(SCRIPT_OUTPUT_DIR / "script_generation_input.json", script_generation_input)
script_package = script_generation_result["data"]["script_package"]

write_json(SCRIPT_OUTPUT_DIR / "script_generation_result.json", script_generation_result)
write_json(SCRIPT_OUTPUT_DIR / "script_package.json", script_package)
write_json(SCRIPT_OUTPUT_DIR / "summary.json", script_generation_result["summary"])
write_jsonl(SCRIPT_OUTPUT_DIR / "llm_calls.jsonl", script_generation_result["llm_calls"])
(SCRIPT_OUTPUT_DIR / "full_script.txt").parent.mkdir(parents=True, exist_ok=True)
(SCRIPT_OUTPUT_DIR / "full_script.txt").write_text(script_package["full_script"], encoding="utf-8")

print(f"Script generation status: {script_generation_result['status']}")
print(f"Sections:                 {script_generation_result['summary']['section_count']}")
print(f"Words:                    {script_generation_result['summary']['total_word_count']}")
print(f"Estimated seconds:        {script_generation_result['summary']['estimated_duration_seconds']}")
print(f"Ready for metadata:   {script_generation_result['handoff']['ready_for_metadata_creation']}")
print(f"Wrote:                    {SCRIPT_OUTPUT_DIR}")


In [ ]:
script_generation_result["summary"]


## App 6: Metadata Creator


In [ ]:
script_result_for_metadata = script_generation_result
if METADATA_SCRIPT_RESULT_PATH:
    external_path = Path(METADATA_SCRIPT_RESULT_PATH)
    script_result_for_metadata = json.loads(external_path.read_text(encoding="utf-8"))
    require_non_empty_script(script_result_for_metadata)
    print(f"Metadata script source: {external_path}")

from src.pipeline.apps.metadata_creator import run as run_metadata_creator

metadata_creator_input = build_metadata_creator_input(RUN_ID, script_result_for_metadata, video_packaging_result)
metadata_creator_result = run_metadata_creator(metadata_creator_input)
require_app_ready(metadata_creator_result, "metadata_creator", handoff_key="ready_for_media_production")
write_json(METADATA_OUTPUT_DIR / "metadata_creator_input.json", metadata_creator_input)
metadata_package = metadata_creator_result["data"]["metadata_package"]
youtube_metadata = metadata_package["youtube_metadata"]

write_json(METADATA_OUTPUT_DIR / "metadata_creator_result.json", metadata_creator_result)
write_json(METADATA_OUTPUT_DIR / "metadata_package.json", metadata_package)
write_json(METADATA_OUTPUT_DIR / "youtube_metadata.json", youtube_metadata)
write_json(METADATA_OUTPUT_DIR / "summary.json", metadata_creator_result["summary"])
write_jsonl(METADATA_OUTPUT_DIR / "llm_calls.jsonl", metadata_creator_result["llm_calls"])

print(f"Metadata status:          {metadata_creator_result['status']}")
print(f"Title options:            {metadata_creator_result['summary']['title_option_count']}")
print(f"Thumbnail options:        {metadata_creator_result['summary']['thumbnail_option_count']}")
print(f"Chapters:                 {metadata_creator_result['summary']['chapter_count']}")
print(f"Source URLs:              {metadata_creator_result['summary']['source_url_count']}")
print(f"Ready for media:   {metadata_creator_result['handoff']['ready_for_media_production']}")
print(f"Wrote:                    {METADATA_OUTPUT_DIR}")


In [ ]:
metadata_creator_result["summary"]
